# 1 · LHS Geometrical Parameter Sampling

**Author:** Héctor Fernández Pinacho  
**Lab:** IDEAL Lab · ETH Zürich

Generates the full propeller design space using constrained Latin Hypercube Sampling (LHS). Each of the 5000 configurations is defined by 17 geometric parameters describing the tip radius, blade count, support ring, and the chord, thickness, camber, max-camber position and pitch angle of the three blade cross-sections (inner, mid, outer). Sampling is constrained so that every design is printable and aerodynamically sensible: minimum absolute wall thickness, solidity limits at the hub and mid stations, chord and thickness taper, monotonic twist, and a per-station angle-of-attack window derived from the inflow angle at the reference RPM.

**Physics inputs:** none (parameter ranges and feasibility constraints are defined in the configuration below)

**Physics outputs:** `csv/01_geometry.csv` — one row per configuration with the 17 geometry parameters

**Structure:**

1. Imports
2. Configuration — all tunable constants, paths and settings
3. Function definitions — feasibility helpers, the LHS generator and the check helper
4. Main code — sample generation, export and validation checks

> **Dead-code audit (2026-07-04):** removed the unused `scale()` helper and the unused `TAPER_RATIO_MIN` / `TAPER_RATIO_MAX` constants (the taper constraint is enforced through the sampling bounds themselves).


## 1. Imports

In [ ]:
import os
import sys

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
from scipy.stats import qmc

import pipeline_config as cfg

## 2. Configuration

In [ ]:
SLIDER_BOUNDS = cfg.SLIDER_BOUNDS

RADIUS_MIN_MM, RADIUS_MAX_MM = SLIDER_BOUNDS["radius_mm"]
RING_HEIGHT_MIN_MM, RING_HEIGHT_MAX_MM = SLIDER_BOUNDS["ring_height_mm"]
RING_THICKNESS_MIN_MM, RING_THICKNESS_MAX_MM = SLIDER_BOUNDS["ring_thickness_mm"]
BLADE_COUNT_MIN, BLADE_COUNT_MAX = SLIDER_BOUNDS["blade_count"]
INNER_THICKNESS_MIN_PCT, INNER_THICKNESS_MAX_PCT = SLIDER_BOUNDS["inner_thickness_pct"]
INNER_MAX_POS_MIN, INNER_MAX_POS_MAX = SLIDER_BOUNDS["inner_max_pos"]
INNER_CAMBER_MIN, INNER_CAMBER_MAX = SLIDER_BOUNDS["inner_camber"]
INNER_CHORD_MIN_MM, INNER_CHORD_MAX_MM = SLIDER_BOUNDS["inner_chord_mm"]
INNER_ANGLE_MIN_DEG, INNER_ANGLE_MAX_DEG = SLIDER_BOUNDS["inner_angle_deg"]
MID_RADIAL_POS_MIN, MID_RADIAL_POS_MAX = SLIDER_BOUNDS["mid_radial_pos"]
MID_CHORD_MIN_MM, MID_CHORD_MAX_MM = SLIDER_BOUNDS["mid_chord_mm"]
MID_ANGLE_MIN_DEG, MID_ANGLE_MAX_DEG = SLIDER_BOUNDS["mid_angle_deg"]
OUTER_THICKNESS_MIN_PCT, OUTER_THICKNESS_MAX_PCT = SLIDER_BOUNDS["outer_thickness_pct"]
OUTER_MAX_POS_MIN, OUTER_MAX_POS_MAX = SLIDER_BOUNDS["outer_max_pos"]
OUTER_CAMBER_MIN, OUTER_CAMBER_MAX = SLIDER_BOUNDS["outer_camber"]
OUTER_CHORD_MIN_MM, OUTER_CHORD_MAX_MM = SLIDER_BOUNDS["outer_chord_mm"]
OUTER_ANGLE_MIN_DEG, OUTER_ANGLE_MAX_DEG = SLIDER_BOUNDS["outer_angle_deg"]

GLOBAL_RANDOM_SEED = cfg.GLOBAL_RANDOM_SEED
RADIUS_BLADE_LHS_SEED = cfg.RADIUS_BLADE_LHS_SEED
GEOMETRY_LHS_SEED = cfg.GEOMETRY_LHS_SEED
N_SAMPLES = cfg.N_SAMPLES

np.random.seed(GLOBAL_RANDOM_SEED)

CSV_DIR = Path("./csv")
OUTPUT_CSV = CSV_DIR / cfg.CSV_NAMES["geometry"]

INNER_STATION_RADIUS_MM = cfg.INNER_ANCHOR_RADIUS_MM
HUB_OUTER_RADIUS_MM = cfg.HUB_STATION_RADIUS_MM

MIN_ABS_WALL_THICKNESS_MM = cfg.MIN_ABS_WALL_THICKNESS_MM
INNER_SOLIDITY_MAX = cfg.INNER_SOLIDITY_MAX
MID_SOLIDITY_MAX = cfg.MID_SOLIDITY_MAX

RPM = cfg.SAMPLING_REFERENCE_RPM
OMEGA_RAD_PER_S = RPM * 2 * np.pi / 60
V_AXIAL_M_PER_S = cfg.SAMPLING_V_AXIAL_M_PER_S
AOA_MIN_DEG = cfg.AOA_MIN_DEG
AOA_MAX_DEG = cfg.AOA_MAX_DEG
ENFORCE_MONOTONIC_TWIST = cfg.ENFORCE_MONOTONIC_TWIST

ANGLE_INTERPOLATION_METHOD = cfg.ANGLE_INTERPOLATION_METHOD
ANGLE_SPLINE_BC_TYPE = cfg.ANGLE_SPLINE_BC_TYPE

## 3. Function Definitions

- **phi_deg(r_mm)** — computes the inflow angle at a given radius: the angle at which the oncoming air meets the rotating blade, from the axial speed and the local tangential speed at the reference RPM. Takes a radius in mm and returns the angle in degrees.
- **aoa_pitch_bounds(r_mm, angle_min_deg, angle_max_deg, pitch_ceil_deg)** — returns the lowest and highest integer pitch angle allowed at a radius so that the local angle of attack stays inside the safe window, optionally capped by an upper pitch ceiling (used to enforce monotonic twist).
- **sample_from_integer_candidates(u, candidates)** — picks one value from a list of feasible integers using a 0–1 sample, giving each candidate an equal share of the unit interval.
- **interpolate_angle_radially(inner_angle_deg, mid_angle_deg, outer_angle_deg, mid_radius_mm, outer_radius_mm, evaluation_radius_mm)** — connects the inner, mid and outer blade angles into a continuous twist along the blade with a natural cubic spline and evaluates it at the requested radius. Returns the interpolated angle in degrees.
- **feasible_inner_angles_for_hub_aoa(mid_angle_deg, outer_angle_deg, mid_radius_mm, outer_radius_mm)** — lists the integer inner angles that keep the angle of attack valid at the hub station and, when monotonic twist is enforced, do not fall below the mid angle. Returns an array of feasible angles.
- **select_inner_angle(lhs_value, mid_angle_deg, outer_angle_deg, mid_radius_mm, outer_radius_mm)** — chooses one inner angle from that feasible list using an LHS sample value. Raises an error if no feasible angle exists.
- **generate_lhs_geometrical_parameters(n_samples)** — the main sampler. Draws the Latin Hypercube designs and, for every design, turns the raw 0–1 samples into the 17 real geometry parameters by floor-binning each sample over its feasibility-narrowed range (solidity-capped chords, wall-thickness-floored thicknesses, taper-capped outer values, AoA-bounded angles) and sizing the support ring from the outer section. Returns the full geometry table as a DataFrame.
- **check(name, series, lo, hi, invert)** — verifies that a column stays inside an expected range and prints a pass/fail line; used by the validation checks at the end of the notebook.


In [ ]:
def phi_deg(r_mm):

    r_m = r_mm / 1000.0
    v_tan = OMEGA_RAD_PER_S * r_m

    return float(np.degrees(np.arctan(V_AXIAL_M_PER_S / v_tan)))


def aoa_pitch_bounds(r_mm, angle_min_deg, angle_max_deg, pitch_ceil_deg=None):

    phi = phi_deg(r_mm)
    lo = int(np.ceil(phi + AOA_MIN_DEG))
    hi = int(np.floor(phi + AOA_MAX_DEG))
    if pitch_ceil_deg is not None:
        hi = min(hi, int(pitch_ceil_deg))
    lo = max(angle_min_deg, lo)
    hi = min(angle_max_deg, hi)
    if hi < lo:
        hi = lo

    return lo, hi


def sample_from_integer_candidates(u, candidates):

    candidates = np.asarray(candidates, dtype=int)
    if len(candidates) == 0:
        raise ValueError("No feasible integer candidates provided.")
    index = min(int(np.floor(u * len(candidates))), len(candidates) - 1)

    return int(candidates[index])


def interpolate_angle_radially(inner_angle_deg, mid_angle_deg, outer_angle_deg, mid_radius_mm, outer_radius_mm, evaluation_radius_mm):

    x = np.array([INNER_STATION_RADIUS_MM, mid_radius_mm, outer_radius_mm], dtype=float)
    y = np.array([inner_angle_deg, mid_angle_deg, outer_angle_deg], dtype=float)
    if not np.all(np.diff(x) > 0):
        raise ValueError("Profile radii must be strictly increasing.")
    if not (x[0] <= evaluation_radius_mm <= x[-1]):
        raise ValueError("Evaluation radius outside radial domain.")
    if ANGLE_INTERPOLATION_METHOD == "natural_cubic_spline_three_profiles":
        spline = CubicSpline(x, y, bc_type=ANGLE_SPLINE_BC_TYPE)
        interpolated = float(spline(evaluation_radius_mm))
    elif ANGLE_INTERPOLATION_METHOD == "linear_three_profiles":
        interpolated = float(np.interp(evaluation_radius_mm, x, y))
    else:
        raise ValueError(f"Unknown interpolation method: {ANGLE_INTERPOLATION_METHOD}")

    return interpolated


def feasible_inner_angles_for_hub_aoa(mid_angle_deg, outer_angle_deg, mid_radius_mm, outer_radius_mm):

    candidates = np.arange(INNER_ANGLE_MIN_DEG, INNER_ANGLE_MAX_DEG + 1)
    hub_angle_values = []
    for candidate_angle in candidates:
        hub_angle_values.append(interpolate_angle_radially(candidate_angle, mid_angle_deg, outer_angle_deg, mid_radius_mm, outer_radius_mm, HUB_OUTER_RADIUS_MM))
    hub_angles = np.array(hub_angle_values)
    hub_aoa = hub_angles - phi_deg(HUB_OUTER_RADIUS_MM)
    feasible = (hub_aoa >= AOA_MIN_DEG) & (hub_aoa <= AOA_MAX_DEG)
    if ENFORCE_MONOTONIC_TWIST:
        feasible &= candidates >= mid_angle_deg

    return candidates[feasible]


def select_inner_angle(lhs_value, mid_angle_deg, outer_angle_deg, mid_radius_mm, outer_radius_mm):

    feasible = feasible_inner_angles_for_hub_aoa(mid_angle_deg, outer_angle_deg, mid_radius_mm, outer_radius_mm)
    if len(feasible) == 0:
        raise ValueError("No feasible inner angle found. Check AoA limits and twist constraints.")

    return sample_from_integer_candidates(lhs_value, feasible)

In [ ]:
def generate_lhs_geometrical_parameters(n_samples):

    radius_blade_samples = qmc.LatinHypercube(d=2, seed=RADIUS_BLADE_LHS_SEED).random(n=n_samples)
    radius_bin_count = int(RADIUS_MAX_MM - RADIUS_MIN_MM + 1)
    blade_count_bin_count = int(BLADE_COUNT_MAX - BLADE_COUNT_MIN + 1)
    radii = (RADIUS_MIN_MM + np.floor(radius_blade_samples[:, 0] * radius_bin_count).clip(0, radius_bin_count - 1)).astype(int)
    blade_counts = (BLADE_COUNT_MIN + np.floor(radius_blade_samples[:, 1] * blade_count_bin_count).clip(0, blade_count_bin_count - 1)).astype(int)
    lhs_samples = qmc.LatinHypercube(d=13, seed=GEOMETRY_LHS_SEED).random(n=n_samples)

    records = []
    for config_id in range(n_samples):
        radius = int(radii[config_id])
        blade_count = int(blade_counts[config_id])

        inner_chord_max = min(int(INNER_SOLIDITY_MAX * 2 * np.pi * HUB_OUTER_RADIUS_MM / blade_count), INNER_CHORD_MAX_MM)
        inner_chord_bin_count = inner_chord_max - INNER_CHORD_MIN_MM + 1
        inner_chord = INNER_CHORD_MIN_MM + int(np.floor(lhs_samples[config_id, 3] * inner_chord_bin_count).clip(0, inner_chord_bin_count - 1))

        inner_thickness_min = max(INNER_THICKNESS_MIN_PCT, int(np.ceil(MIN_ABS_WALL_THICKNESS_MM * 100 / inner_chord)))
        inner_thickness_bin_count = INNER_THICKNESS_MAX_PCT - inner_thickness_min + 1
        inner_thickness = inner_thickness_min + int(np.floor(lhs_samples[config_id, 0] * inner_thickness_bin_count).clip(0, inner_thickness_bin_count - 1))

        inner_max_pos = INNER_MAX_POS_MIN + int(np.floor(lhs_samples[config_id, 1] * (INNER_MAX_POS_MAX - INNER_MAX_POS_MIN + 1)).clip(0, INNER_MAX_POS_MAX - INNER_MAX_POS_MIN))
        inner_camber = INNER_CAMBER_MIN + int(np.floor(lhs_samples[config_id, 2] * (INNER_CAMBER_MAX - INNER_CAMBER_MIN + 1)).clip(0, INNER_CAMBER_MAX - INNER_CAMBER_MIN))

        mid_radial_pos_bin_count = 5
        mid_radial_pos = round(MID_RADIAL_POS_MIN + int(np.floor(lhs_samples[config_id, 5] * mid_radial_pos_bin_count).clip(0, mid_radial_pos_bin_count - 1)) * 0.1, 1)

        mid_radius = mid_radial_pos * radius
        mid_chord_hi = min(int(MID_SOLIDITY_MAX * 2 * np.pi * mid_radius / blade_count), MID_CHORD_MAX_MM)
        mid_chord_bin_count = mid_chord_hi - MID_CHORD_MIN_MM + 1
        mid_chord = MID_CHORD_MIN_MM + int(np.floor(lhs_samples[config_id, 6] * mid_chord_bin_count).clip(0, mid_chord_bin_count - 1))

        mid_angle_lo, mid_angle_hi = aoa_pitch_bounds(mid_radius, MID_ANGLE_MIN_DEG, MID_ANGLE_MAX_DEG)
        mid_angle_bin_count = mid_angle_hi - mid_angle_lo + 1
        mid_angle = mid_angle_lo + int(np.floor(lhs_samples[config_id, 7] * mid_angle_bin_count).clip(0, mid_angle_bin_count - 1))

        outer_chord_hi = max(OUTER_CHORD_MIN_MM, min(mid_chord, OUTER_CHORD_MAX_MM))
        outer_chord_bin_count = outer_chord_hi - OUTER_CHORD_MIN_MM + 1
        outer_chord = OUTER_CHORD_MIN_MM + int(np.floor(lhs_samples[config_id, 8] * outer_chord_bin_count).clip(0, outer_chord_bin_count - 1))

        outer_thickness_min = max(OUTER_THICKNESS_MIN_PCT, int(np.ceil(MIN_ABS_WALL_THICKNESS_MM * 100 / outer_chord)))
        outer_thickness_high = min(inner_thickness, OUTER_THICKNESS_MAX_PCT)
        outer_thickness_bin_count = outer_thickness_high - outer_thickness_min + 1
        outer_thickness = outer_thickness_min + int(np.floor(lhs_samples[config_id, 9] * outer_thickness_bin_count).clip(0, outer_thickness_bin_count - 1))

        outer_max_pos = OUTER_MAX_POS_MIN + int(np.floor(lhs_samples[config_id, 11] * (OUTER_MAX_POS_MAX - OUTER_MAX_POS_MIN + 1)).clip(0, OUTER_MAX_POS_MAX - OUTER_MAX_POS_MIN))
        outer_camber = OUTER_CAMBER_MIN + int(np.floor(lhs_samples[config_id, 10] * (OUTER_CAMBER_MAX - OUTER_CAMBER_MIN + 1)).clip(0, OUTER_CAMBER_MAX - OUTER_CAMBER_MIN))

        if ENFORCE_MONOTONIC_TWIST:
            outer_pitch_ceil = mid_angle
        else:
            outer_pitch_ceil = None
        outer_angle_lo, outer_angle_hi = aoa_pitch_bounds(radius, OUTER_ANGLE_MIN_DEG, OUTER_ANGLE_MAX_DEG, pitch_ceil_deg=outer_pitch_ceil)
        outer_angle_bin_count = outer_angle_hi - outer_angle_lo + 1
        outer_angle = outer_angle_lo + int(np.floor(lhs_samples[config_id, 12] * outer_angle_bin_count).clip(0, outer_angle_bin_count - 1))

        inner_angle = select_inner_angle(lhs_samples[config_id, 4], mid_angle, outer_angle, mid_radius, radius)

        outer_angle_rad = np.radians(outer_angle)
        min_ring_height = (outer_chord * np.sin(outer_angle_rad) + outer_thickness * np.sin(outer_angle_rad) / 6 + outer_camber * np.sin(outer_angle_rad) / 3)
        ring_height = float(np.clip(np.ceil(min_ring_height + 1), RING_HEIGHT_MIN_MM, RING_HEIGHT_MAX_MM))
        ring_thickness = float(np.clip(ring_height / 2, RING_THICKNESS_MIN_MM, RING_THICKNESS_MAX_MM))

        records.append({
            "config_id":           config_id,
            "radius_mm":           radius,
            "ring_height_mm":      ring_height,
            "ring_thickness_mm":   ring_thickness,
            "blade_count":         blade_count,
            "inner_thickness_pct": inner_thickness,
            "inner_max_pos":       inner_max_pos,
            "inner_camber":        inner_camber,
            "inner_chord_mm":      inner_chord,
            "inner_angle_deg":     inner_angle,
            "mid_radial_pos":      mid_radial_pos,
            "mid_chord_mm":        mid_chord,
            "mid_angle_deg":       mid_angle,
            "outer_thickness_pct": outer_thickness,
            "outer_max_pos":       outer_max_pos,
            "outer_camber":        outer_camber,
            "outer_chord_mm":      outer_chord,
            "outer_angle_deg":     outer_angle,
        })

    columns = ["config_id", "radius_mm", "ring_height_mm", "ring_thickness_mm", "blade_count", "inner_thickness_pct", "inner_max_pos", "inner_camber", "inner_chord_mm", "inner_angle_deg", "mid_radial_pos", "mid_chord_mm", "mid_angle_deg", "outer_thickness_pct", "outer_max_pos", "outer_camber", "outer_chord_mm", "outer_angle_deg"]

    return pd.DataFrame(records, columns=columns)

In [ ]:
def check(name, series, lo, hi=None, invert=False):

    global all_ok
    vmin = round(float(series.min()), 3)
    vmax = round(float(series.max()), 3)
    if invert:
        ok = (series >= lo).all()
        expected = f">= {lo}"
    else:
        ok = (series >= lo).all() and (series <= hi).all()
        expected = f"[{lo}, {hi}]"
    status = "  OK  " if ok else " FAIL "
    print(f"  [{status}]  {name:<38} actual [{vmin}, {vmax}]  expected {expected}")
    if not ok:
        all_ok = False

## 4. Main Code

The main code first fixes the configuration: the slider bounds of the Propeller Configurator, the random seeds, the feasibility constants (wall thickness, solidity, taper, angle-of-attack window) and the reference operating point used to derive the inflow angle. It then draws the 5000 constrained LHS designs, saves them to `csv/01_geometry.csv`, and finally verifies every parameter range, the solidity and printability limits, and the twist monotonicity before the CSV is trusted by the downstream notebooks.


### 4.1 Generate and Export

In [ ]:
geometry_df = generate_lhs_geometrical_parameters(N_SAMPLES)

CSV_DIR.mkdir(parents=True, exist_ok=True)
geometry_df.to_csv(OUTPUT_CSV, index=False)

print(f"Generated  : {len(geometry_df)} configurations")
print(f"Columns    : {list(geometry_df.columns)}")
print(f"Saved to   : {OUTPUT_CSV}")
geometry_df.head()

### 4.2 Validation Checks

Verifies that every geometric parameter stays inside its slider bounds, that the hub and mid solidity limits hold, that the absolute wall thickness is printable at all three stations, and that the twist decreases monotonically from root to tip.


In [ ]:
all_ok = True

df = geometry_df
print("Slider bounds")
print("-" * 80)
check("radius_mm", df["radius_mm"], RADIUS_MIN_MM, RADIUS_MAX_MM)
check("ring_height_mm", df["ring_height_mm"], RING_HEIGHT_MIN_MM, RING_HEIGHT_MAX_MM)
check("ring_thickness_mm", df["ring_thickness_mm"], RING_THICKNESS_MIN_MM, RING_THICKNESS_MAX_MM)
check("blade_count", df["blade_count"], BLADE_COUNT_MIN, BLADE_COUNT_MAX)
check("inner_chord_mm", df["inner_chord_mm"], INNER_CHORD_MIN_MM, INNER_CHORD_MAX_MM)
check("inner_angle_deg", df["inner_angle_deg"], INNER_ANGLE_MIN_DEG, INNER_ANGLE_MAX_DEG)
check("inner_thickness_pct", df["inner_thickness_pct"], INNER_THICKNESS_MIN_PCT, INNER_THICKNESS_MAX_PCT)
check("inner_camber", df["inner_camber"], INNER_CAMBER_MIN, INNER_CAMBER_MAX)
check("inner_max_pos", df["inner_max_pos"], INNER_MAX_POS_MIN, INNER_MAX_POS_MAX)
check("mid_radial_pos", df["mid_radial_pos"], MID_RADIAL_POS_MIN, MID_RADIAL_POS_MAX)
check("mid_chord_mm", df["mid_chord_mm"], MID_CHORD_MIN_MM, MID_CHORD_MAX_MM)
check("mid_angle_deg", df["mid_angle_deg"], MID_ANGLE_MIN_DEG, MID_ANGLE_MAX_DEG)
check("outer_chord_mm", df["outer_chord_mm"], OUTER_CHORD_MIN_MM, OUTER_CHORD_MAX_MM)
check("outer_angle_deg", df["outer_angle_deg"], OUTER_ANGLE_MIN_DEG, OUTER_ANGLE_MAX_DEG)
check("outer_thickness_pct", df["outer_thickness_pct"], OUTER_THICKNESS_MIN_PCT, OUTER_THICKNESS_MAX_PCT)
check("outer_camber", df["outer_camber"], OUTER_CAMBER_MIN, OUTER_CAMBER_MAX)
check("outer_max_pos", df["outer_max_pos"], OUTER_MAX_POS_MIN, OUTER_MAX_POS_MAX)

print()
print("Solidity")
print("-" * 80)
mid_r_mm = df["mid_radial_pos"] * df["radius_mm"]
sol_inner = df["inner_chord_mm"] * df["blade_count"] / (2 * np.pi * HUB_OUTER_RADIUS_MM)
sol_mid = df["mid_chord_mm"] * df["blade_count"] / (2 * np.pi * mid_r_mm)
check("inner_solidity (at hub radius)", sol_inner, 0, INNER_SOLIDITY_MAX)
check("mid_solidity", sol_mid, 0, MID_SOLIDITY_MAX)

print()
print("Printability (abs wall thickness >= 1 mm)")
print("-" * 80)
inner_abs = df["inner_chord_mm"] * df["inner_thickness_pct"] / 100
outer_abs = df["outer_chord_mm"] * df["outer_thickness_pct"] / 100
mid_t_pct = df["inner_thickness_pct"] + df["mid_radial_pos"] * (df["outer_thickness_pct"] - df["inner_thickness_pct"])
mid_abs = df["mid_chord_mm"] * mid_t_pct / 100
check("inner_abs_thickness_mm", inner_abs, MIN_ABS_WALL_THICKNESS_MM, invert=True)
check("mid_abs_thickness_mm", mid_abs, MIN_ABS_WALL_THICKNESS_MM, invert=True)
check("outer_abs_thickness_mm", outer_abs, MIN_ABS_WALL_THICKNESS_MM, invert=True)

print()
print("Twist monotonicity")
print("-" * 80)
mono = (df["inner_angle_deg"] >= df["mid_angle_deg"]) & (df["mid_angle_deg"] >= df["outer_angle_deg"])
status = "  OK  " if mono.all() else " FAIL "
print(f"  [{status}]  {mono.sum()}/{len(df)} configs have monotonic twist")
if not mono.all():
    all_ok = False

print()
print("=" * 80)
print("ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED — review output above")
print("=" * 80)